# Sincronização: Google Colab, Drive e GitHub

1. No Colab, crie o segredo `GITHUB_TOKEN` e permita o acesso deste notebook. Consulte o README para a configuração inicial.
2. Salve e feche outros editores desta cópia e execute a célula abaixo no início de cada sessão. Autorize o Drive quando solicitado.
3. Abra os notebooks a partir de `Meu Drive/vision-transfer-learning`, salve suas alterações e mantenha esta sessão ativa para sincronizá-las.

O monitor envia alterações estáveis automaticamente. Se a `main` remota mudar durante o trabalho, ele para e informa como proceder. Confira as mensagens desta célula; em caso de conflito, nenhuma resolução é feita automaticamente.


In [1]:
from pathlib import Path
import importlib.util
import os
import subprocess
import tempfile
import threading

try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("Esta célula é para o Colab. Localmente, use: uv run --no-sync python sync.py") from exc

REPO_URL = "https://github.com/Gengo250/vision-transfer-learning.git"
REPO_DIR = Path("/content/drive/MyDrive/vision-transfer-learning")
LOCK_DIR = REPO_DIR.parent / f".{REPO_DIR.name}.sync-lock"

drive.mount("/content/drive")
try:
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError("segredo vazio")
except Exception as exc:
    raise RuntimeError("Configure GITHUB_TOKEN nos Segredos do Colab e permita acesso a este notebook.") from exc

if not REPO_DIR.exists():
    try:
        LOCK_DIR.mkdir()
    except FileExistsError as exc:
        raise RuntimeError(f"Sincronização em andamento ou bloqueio antigo: {LOCK_DIR}") from exc
    try:
        if not REPO_DIR.exists():
            print("Clonando o repositório no Drive...")
            # O token fica somente no ambiente do processo e em memória.
            with tempfile.TemporaryDirectory(prefix="vision-clone-") as temporary:
                helper = Path(temporary) / "askpass"
                helper.write_text('#!/bin/sh\ncase "$1" in *sername*) echo x-access-token;; *) printf "%s\\n" "$VISION_SYNC_TOKEN";; esac\n')
                helper.chmod(0o700)
                clone_env = os.environ.copy()
                clone_env.update(GIT_ASKPASS=str(helper), GIT_TERMINAL_PROMPT="0",
                                 VISION_SYNC_TOKEN=token, GIT_CONFIG_COUNT="1",
                                 GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="")
                subprocess.run(["git", "-c", "http.followRedirects=false", "clone", "--branch", "main",
                                REPO_URL, str(REPO_DIR)], check=True, env=clone_env)
                del clone_env
    finally:
        LOCK_DIR.rmdir()

if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} não contém um clone Git completo. Preserve e revise a pasta antes de continuar.")

if not (REPO_DIR / "sync.py").is_file():
    raise RuntimeError("A cópia do Drive é antiga e ainda não tem sync.py. Use o diagnóstico e a atualização segura em COLAB_RECUPERACAO.md, a partir de um notebook separado.")

if "sync_thread" in globals() and sync_thread.is_alive():
    del token
    print("O monitor desta sessão já está ativo.")
else:
    spec = importlib.util.spec_from_file_location("vision_sync", REPO_DIR / "sync.py")
    sync_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sync_module)
    ready = threading.Event()
    sync_request = threading.Event()
    result = []
    def iniciar_monitor(secret):
        try:
            sync_module.watch(secret, colab=True, ready=ready, wake=sync_request)
        except Exception as exc:
            result.append(str(exc).replace(secret, "[TOKEN OCULTO]"))
            print(f"Sincronização interrompida: {result[-1]}", flush=True)
            ready.set()
    sync_thread = threading.Thread(target=iniciar_monitor, args=(token,), daemon=True)
    sync_thread.start()
    del token
    if not ready.wait(120):
        print("A inicialização ainda está em andamento; confira as mensagens desta célula.")
    elif result:
        raise RuntimeError(result[-1])

os.chdir(REPO_DIR)
print(f"Diretório do projeto: {REPO_DIR}")


Mounted at /content/drive
Repositório conferido; nenhuma atualização remota pendente.
Monitor ativo: verifica alterações estáveis a cada 60 segundos. Salve os arquivos antes de sincronizar.
Diretório do projeto: /content/drive/MyDrive/vision-transfer-learning


## Solicitar publicação no GitHub

Depois de salvar os arquivos no Drive, execute a célula para solicitar uma verificação antecipada. O monitor aguarda os arquivos ficarem estáveis e aplica as mesmas verificações da sincronização automática. Não é necessário executar esta célula para o envio automático.


In [2]:
if "sync_thread" not in globals() or not sync_thread.is_alive():
    raise RuntimeError("O monitor não está ativo. Execute a célula inicial e confira as mensagens.")
sync_request.set()
print("Verificação solicitada. O monitor aguarda os arquivos ficarem estáveis antes de enviar.")


Verificação solicitada. O monitor aguarda os arquivos ficarem estáveis antes de enviar.


## Verificar dependências do notebook de treinamento

O Colab usa seu próprio runtime. A célula abaixo instala `jupyterlab` apenas se faltar, pois o notebook de treinamento atual o importa. A instalação é temporária e não altera a configuração `uv` local.


In [3]:
import importlib.util
import subprocess
import sys

for nome in ("torch", "torchvision", "numpy", "pandas", "matplotlib", "sklearn", "PIL", "jupyterlab", "ipykernel"):
    print(f"{nome}: {'disponível' if importlib.util.find_spec(nome) else 'ausente'}")
if importlib.util.find_spec("jupyterlab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "jupyterlab"], check=True)


torch: disponível
torchvision: disponível
numpy: disponível
pandas: disponível
matplotlib: disponível
sklearn: disponível
PIL: disponível
jupyterlab: ausente
ipykernel: disponível
